# VSD Cohort Laterality Audit with Intrinsic Bone Anatomy

> **Superseded diagnostic:** the X-only rule in this notebook is not rotation-stable. Use `08_vsd_focused_ct_stl_target_overlay_audit.ipynb` and `vsd_cohort_laterality_multibone_v2` for the authoritative Stage 1 laterality verdict.

Absolute LPS X is not a safe side test after separately cropped knees have been re-centered. This audit instead compares physical fibula and tibia centroids within each target: the fibula is lateral to the tibia, so fibula X greater than tibia X indicates a Left knee and lower indicates a Right knee in LPS.

The existing unversioned per-bone targets are used only to detect candidate mismatches. Any override must be confirmed visually and replayed in the versioned v1 target pipeline.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name != "TestProject":
    ROOT = ROOT / "TestProject"
GT_ROOT = ROOT / "data" / "interim" / "gt_per_bone_256" / "healthy"
REPORT_DIR = ROOT / "reports" / "manifests"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
BONES = ("tibia", "fibula")
SELECTED_VISUAL_REVIEW = [
    "VSD_001_Left",
    "VSD_010_Left", "VSD_010_Right",
    "VSD_015_Left",
    "VSD_z001_Left",
    "VSD_z023_Left",
    "VSD_z036_Right",
    "VSD_z050_Right",
    "VSD_z063_Right",
]

In [ ]:
def mask_centroid_world(path):
    image = nib.load(str(path))
    assert nib.aff2axcodes(image.affine) == ("L", "P", "S")
    mask = np.asarray(image.dataobj) > 0
    assert mask.any(), f"empty mask: {path}"
    centroid_voxel = np.argwhere(mask).mean(axis=0)
    return nib.affines.apply_affine(image.affine, centroid_voxel)


rows = []
for case_dir in sorted(p for p in GT_ROOT.iterdir() if p.is_dir() and p.name.startswith("VSD_")):
    key = case_dir.name
    named_side = key.rsplit("_", 1)[1]
    tibia = mask_centroid_world(case_dir / f"{key}_tibia.nii.gz")
    fibula = mask_centroid_world(case_dir / f"{key}_fibula.nii.gz")
    delta_x = float(fibula[0] - tibia[0])
    intrinsic_side = "Left" if delta_x > 0 else "Right"
    rows.append({
        "sample_id": key,
        "named_side": named_side,
        "intrinsic_side": intrinsic_side,
        "fibula_minus_tibia_lps_x_mm": delta_x,
        "status": "PASS" if named_side == intrinsic_side else "REVIEW_REQUIRED",
        "source": "legacy_gt_per_bone_256_preliminary_audit",
        "selected_visual_review": key in SELECTED_VISUAL_REVIEW,
    })

audit = pd.DataFrame(rows).sort_values("sample_id").reset_index(drop=True)
assert len(audit) == 58
audit.to_csv(REPORT_DIR / "vsd_cohort_laterality_intrinsic_v1.csv", index=False)
flagged = audit[audit.status == "REVIEW_REQUIRED"]
summary = {
    "audit_version": "vsd_cohort_laterality_intrinsic_v1",
    "method": "fibula_vs_tibia_centroid_in_LPS",
    "total": len(audit),
    "pass": int((audit.status == "PASS").sum()),
    "review_required": len(flagged),
    "flagged_samples": flagged.sample_id.tolist(),
    "vsd010_status": audit[audit.sample_id.str.startswith("VSD_010_")][["sample_id", "status"]].to_dict("records"),
    "tkr_survivor_status": audit[audit.sample_id.isin(["VSD_z050_Right", "VSD_z063_Right"])][["sample_id", "status"]].to_dict("records"),
    "limitation": "Preliminary audit on legacy targets; confirm flagged cases visually and replay versioned targets before certification.",
}
(REPORT_DIR / "vsd_cohort_laterality_intrinsic_v1.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for axis, key in zip(axes.ravel(), SELECTED_VISUAL_REVIEW):
    case_dir = GT_ROOT / key
    tib_img = nib.load(str(case_dir / f"{key}_tibia.nii.gz"))
    fib_img = nib.load(str(case_dir / f"{key}_fibula.nii.gz"))
    tib = np.asarray(tib_img.dataobj) > 0
    fib = np.asarray(fib_img.dataobj) > 0
    overlay = np.zeros((*tib.shape[:2], 3), dtype=np.float32)
    overlay[..., 1] = np.max(tib, axis=2)
    overlay[..., 0] = np.max(fib, axis=2)
    row = audit.loc[audit.sample_id == key].iloc[0]
    axis.imshow(np.transpose(overlay, (1, 0, 2)), origin="lower")
    axis.set_title(
        f"{key}\nintrinsic={row.intrinsic_side}, dx={row.fibula_minus_tibia_lps_x_mm:.1f} mm\n{row.status}",
        color="red" if row.status == "REVIEW_REQUIRED" else "black",
        fontsize=9,
    )
    axis.axis("off")
fig.suptitle("VSD laterality review: tibia=green, fibula=red", fontsize=14)
fig.tight_layout()
qa_path = REPORT_DIR / "vsd_laterality_selected_review_v1.png"
fig.savefig(qa_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"QA figure: {qa_path}")

## Decision gate

Do not apply a VSD010 mapping override: both VSD010 knees pass the intrinsic anatomy test. Review the selected figure, with special attention to VSD_z023_Left and VSD_z036_Right. Only confirmed mismatches should enter a non-destructive mapping file. Raw filenames remain unchanged.